
### Student Information

GitHub ID: s1234560108-netizen

Kaggle name: [Fill in my actual Kaggle username]

Kaggle private scoreboard snapshot:

Not available. I did not submit predictions before the competition closed on November 22, 2026. Therefore, I have no official private-leaderboard score or ranking.

The screenshot below documents the closed competition page, not a leaderboard ranking.

![Competition closed - no official ranking](./pics/pic_ranking.png)


---

# **Instructions**

This lab has **three parts**: exercises in the Master notebook, exercises and an agent conversation in the AgentPipeline notebook, and an internal Kaggle competition documented in this Homework notebook. Together they add up to **100 points**.

**Environment recommendations to solve lab 2:**
- **Part 1 and 2 exercises:** Everything can be run with only CPU.
- **Part 3 exercises:** For the competition you will probably need GPU to train your models, so it is recommended to use Colab or Kaggle if you don't have a laptop with a dedicated GPU.

## **Part 1 (40 pts):**

1. __Main Exercises (35 pts):__ Do the **take home exercises** from Sections: `1. Data Preparation` to `8. Information Retrieval and RAG`, in the DM2026-Lab2-Master Notebook. Total: `13 exercises`. Commit your code and submit the repository link to NTU Cool.

2. **Code Comments (5 pts):** **Tidy up the code in your notebook**. 

## **Part 2 (30 pts):**

1. **Agent Prompt (20 pts):** Interact with the Agent using appropriate prompts to guide it through and complete all the required tasks.

2. **Main Exercises (10 pts):** Do the remaining **take home exercises** from DM2026-Lab2-AgentPipeline Notebook. Total: `3 exercises required`. Commit your code and submit the repository link to NTU Cool.

## **Part 3 (30 pts):**

1. **Kaggle Competition Participation (15 pts):** Submit your predictions to the [Data Mining Class Kaggle Competition](https://www.kaggle.com/t/d99def2e62194b0d8f633267c32e6d1e) by **November 22**. Check Kaggle for the exact closing time. The competition concerns emotion recognition on Twitter; participation points use the Private Leaderboard ranking:
    - If your submission meets or exceeds the **baseline score (F1 score: 0.5)**, and there are no    significant issues with your report or code, you will automatically receive a base score of 5 points of the 15 points.

        The remaining 10 points will be determined by your ranking among participants who meet or exceed the baseline:

        |Ranking  |Additional Points  |
        | --- | --- |
        |Top 10% |+10 |
        |Top 20% |+9 |
        |Top 30% |+8 |
        |Top 40% |+7 |
        |Top 50% |+6 |
        |Top 60% |+5 |
        |Top 70% |+4 |
        |Top 80% |+3 |
        |Top 90% |+2 |
        |<Top 90% |+1 |

    - If your submission does not reach the baseline score, your grade will be lower than 5. The score will range from 0 to 4 points, depending on your Private Leaderboard ranking and the relative performance of students whose submissions are also below the baseline.

2. **Competition Report (15 pts)** A report section to be filled in inside this notebook in Markdown Format, we already provided you with the template below. You need to describe your work developing the model for the competition. The report should include a section describing briefly the following elements: 
* All of your AI prompts and what you decide do use.
* Your preprocessing steps.
* The feature engineering steps.
* Explanation of your model.

[Markdown Guide - Basic Syntax](https://www.markdownguide.org/basic-syntax/)

**`Things to note for Part 3:`**

* **The code used for the competition should be in this Jupyter Notebook File** `DM2026-Lab2-Homework.ipynb`.

* **Push the code used for the competition to your repository**.

* **The code should have a clear separation for the same sections of the report, AI prompts, preprocessing, feature engineering and model explanation. Briefly comment your code for easier understanding, we provide a template at the end of this notebook.**

* Save a screenshot of your Kaggle Private Leaderboard ranking as `pics/pic_ranking.png` and keep the image link in the Student Information cell at the top of this notebook.

The Kaggle competition closes on **November 22**. The three days afterward are for completing this notebook's Markdown report, checking your files, and submitting the lab by **November 25 at 11:59 pm (Wednesday)**.

Push your work to your repository, then submit its link and a ZIP on the corresponding NTU COOL assignment. The ZIP must contain `DM2026-Lab2-Master.ipynb`, `DM2026-Lab2-AgentPipeline.ipynb`, `DM2026-Lab2-Homework.ipynb`, both encrypted logs in `session_logs/`, and `pics/pic_ranking.png`. Name it with your student ID and your name. The two logs are git-ignored, so they are not in your repository and the ZIP is the only copy that reaches us: open it before uploading and check both are inside. If you miss them, you can upload them on NTU COOL until **November 26 at 11:59 pm**. See `README.md` for the exact file paths and setup steps.

---

Next you will find the template report with some simple markdown syntax explanations, use it to structure your content.

You can delete the syntax suggestions after you use them.

---

# Project Report

## 1. Model Development (10 pts Required)

### 1.1 Preprocessing Steps

My goal in the Kaggle part of Lab 2 was to build a reproducible classifier for six emotions in Twitter posts: **anger, disgust, fear, joy, sadness, and surprise**.

I began by loading four files: `final_posts.json`, `data_identification.csv`, `emotion.csv`, and `sampleSubmission.csv`. I merged the tweet texts, training/test identifiers, and emotion labels using `post_id` rather than assuming that the files had the same row order. I also verified that the unlabeled test IDs matched those in the sample submission.

After merging the data, I obtained **4,800 tweets**, including **3,360 labeled training examples** and **1,440 unlabeled competition test examples**.

The training labels were imbalanced. Joy was the largest class with 875 examples, followed by anger (674), sadness (628), disgust (454), surprise (441), and fear (288). Therefore, I chose **macro-F1** as my primary local evaluation metric, since it gives equal importance to each emotion category regardless of its frequency. I also examined accuracy and per-class precision, recall, and F1.

To evaluate my models, I used `StratifiedGroupKFold` with five folds, shuffling, and `random_state=42`. I grouped tweets based on their text after removing surrounding whitespace and converting it to lowercase. I selected the first split, which contained **2,687 training tweets** and **673 validation tweets**.

I verified that no identical normalized text appeared in both partitions. This reduced the risk of exact-duplicate leakage, although it could not eliminate all possible near-duplicates.

I retained the original tweet text rather than aggressively removing punctuation, hashtags, and other potentially useful expressions. More importantly, I fitted the TF-IDF vectorizers only on the training partition. The validation set was transformed using the fitted vectorizers without influencing the vocabulary or IDF weights.

This allowed me to evaluate the models on data that had not been used during feature fitting or classifier training.

### 1.2 Feature Engineering Steps

I compared two TF-IDF representations to examine how different text features affected emotion classification.

**Word-level TF-IDF**

I first used word-level TF-IDF as my baseline representation. The vectorizer included word unigrams and bigrams, with `ngram_range=(1, 2)`, `min_df=2`, `max_df=0.95`, `max_features=30000`, Unicode accent stripping, and `sublinear_tf=True`.

This produced **6,012 features**.

I selected word-level TF-IDF because it is simple, interpretable, and widely used for text classification. Unigrams represent individual words, while bigrams can preserve limited local context, such as the expression “not happy.”

However, I recognized that Twitter posts often contain informal language, abbreviations, spelling variations, and short expressions. Word-level TF-IDF may have difficulty capturing similarities among these variations.

**Character-level TF-IDF**

I then tested a character-level TF-IDF representation using `analyzer="char_wb"`, `ngram_range=(3, 5)`, `min_df=2`, `max_features=60000`, and `sublinear_tf=True`.

This produced **23,360 features**.

Unlike word-level TF-IDF, character n-grams represent smaller fragments of words. I expected these features to be useful when different words shared similar spellings or when tweets contained informal variations.

For example, related spellings may share character sequences even when they are treated as separate words by a word-level vectorizer.

Both representations were evaluated using the same training and validation split, which made the model scores directly comparable.

However, I also changed the SVM regularization parameter between the two configurations. Therefore, the performance difference reflects the combined effect of the feature representation and classifier setting rather than the isolated effect of character n-grams.

### 1.3 Explanation of My Model

I chose Linear Support Vector Classification (`LinearSVC`) because it is efficient for high-dimensional sparse text representations and provides a straightforward baseline for multiclass classification.

I evaluated two model configurations:

- Word-level TF-IDF with LinearSVC (`C=1.0`)
- Character-level TF-IDF with LinearSVC (`C=0.25`)

Both used `random_state=42` and `max_iter=5000`.

**Validation results**

| Model | Accuracy | Macro-F1 |
|---|---:|---:|
| Word TF-IDF + LinearSVC | 0.4666 | 0.4422 |
| Character TF-IDF + LinearSVC | **0.5557** | **0.5342** |

The second configuration improved validation macro-F1 from **0.4422 to 0.5342**, an increase of **0.0920**, or 9.20 percentage points. Accuracy also increased by approximately 8.92 percentage points.

To understand whether the improvement was consistent across emotion categories, I compared the per-class F1 scores.

| Emotion | Word TF-IDF F1 | Character TF-IDF F1 |
|---|---:|---:|
| Anger | 0.4414 | 0.5149 |
| Disgust | 0.3273 | 0.3731 |
| Fear | 0.4211 | 0.5934 |
| Joy | 0.5729 | 0.6717 |
| Sadness | 0.4861 | 0.5214 |
| Surprise | 0.4048 | 0.5309 |

The second configuration achieved higher F1 scores for all six emotions, especially fear and surprise.

However, I also found that **disgust remained a difficult category**. Although its F1 increased, its recall decreased from 0.2967 to 0.2747. At the same time, its precision increased from 0.3649 to 0.5814.

This showed me that a higher F1 score does not necessarily mean that both precision and recall have improved. I therefore considered the complete classification report rather than looking only at the overall macro-F1.

The original confusion matrix also showed that the model frequently confused related negative emotions, including disgust and anger. This suggests that separating emotionally similar categories remains challenging for a linear classifier based on text features.

**Final model selection**

Based on the observed validation performance, I selected the character-level TF-IDF configuration with `LinearSVC(C=0.25)`.

After making this decision, I refitted the TF-IDF vectorizer and classifier using all **3,360 labeled training examples**. The final vectorizer contained **26,521 features**.

I then generated predictions for all **1,440 unlabeled Kaggle test examples**.

The prediction file, `kaggle_submission_char_svm.csv`, contains the required `post_id` and `label` columns. I verified that the IDs appeared in the same order as the sample submission, that all labels were valid, and that no predictions were missing.

The predicted class distribution was joy (527), anger (345), sadness (247), surprise (150), disgust (99), and fear (72).

**Limitations and future improvements**

My reported macro-F1 of **0.5342 is a local validation score**, not a Kaggle Private Leaderboard result.

Although the second model performed better on my validation set, this does not guarantee that it would achieve the same performance on the hidden competition test set.

I also evaluated only two model configurations using one grouped validation split. Since the representation and SVM regularization parameter changed together, I cannot conclude that character-level features alone caused the improvement.

For a more rigorous evaluation, I would consider grouped cross-validation, controlled parameter comparisons, and additional analysis of misclassified tweets. I would also investigate class-weighted models to see whether they could improve minority-class performance without sacrificing overall macro-F1.

I did not run these additional experiments in the results reported here.

### 1.4 AI Prompts, Assistance, and My Decisions

I used ChatGPT as an assistant during my Kaggle workflow, primarily for coding guidance, troubleshooting, discussing possible modeling approaches, and interpreting experimental results.

My interaction with AI was iterative. Instead of asking for an entire solution and accepting it immediately, I executed each step in JupyterLab, shared the actual outputs, and used those results to decide what to do next.

The following table records the main prompts and execution results I provided to ChatGPT, together with the suggestions I adopted. The quoted messages are drawn from my actual interactions, with longer outputs summarized here; the full numerical outputs are preserved in my Notebook.

| Prompt or information I provided to AI | My decision and reason |
|---|---|
| “我不知道怎麼新增一個code cell” | I followed the JupyterLab instructions to insert and execute code cells so that all competition code remained in the required Homework Notebook. |
| “NOT FOUND: final_posts.json ... Files found: 0/4” | I checked the file locations and uploaded the four original Kaggle files into the project's `data/` directory. |
| “FOUND: final_posts.json ... Files found: 4/4” | I proceeded with data preparation only after verifying that all required files were accessible. |
| “Total tweets: 4800 Training samples: 3360 Test samples: 1440” | I checked the merged dataset and class distribution. I used macro-F1 because of class imbalance and kept the unlabeled Kaggle test set separate from model development. |
| “Training samples: 2687 Validation samples: 673 Duplicate text overlap: 0” | I retained the grouped, approximately stratified split after checking the sample counts and confirming that identical normalized texts did not cross partitions. |
| “Training feature matrix: (2687, 6012) ... Vocabulary size: 6012” | I used word-level TF-IDF to establish a reproducible baseline before trying a more complex representation. |
| “Validation Accuracy: 0.4666 Validation Macro-F1: 0.4422” | I reviewed the classification report and confusion matrix. Because performance was weak for several classes, I decided to test an alternative feature representation. |
| “Character TF-IDF training shape: (2687, 23360) ... Vocabulary size: 23360” | I adopted the suggestion to evaluate character n-grams because they could help capture spelling variations and word fragments in informal tweets. |
| “Validation Accuracy: 0.5557 Validation Macro-F1: 0.5342” | I verified the improvement using the actual model output. I also examined the per-class results and noticed that disgust recall had decreased despite its higher F1. |
| “Final training samples: 3360 Kaggle test predictions: 1440 Final vocabulary size: 26521” | I checked the final prediction file's dimensions, label distribution, and ID order instead of assuming that successful file generation guaranteed a valid competition result. |

**How I evaluated AI suggestions**

The AI suggested trying character-level TF-IDF after I obtained a relatively low baseline macro-F1. I decided to use this suggestion because it had a reasonable connection to the characteristics of Twitter text, and I could evaluate it using the same validation split.

I did not accept the predicted improvement as a fact. I executed the new feature-engineering and classification code, then checked the resulting accuracy, macro-F1, and per-class report.

After observing the improvement, I selected the character-level configuration as my final model.

I also decided not to perform extensive additional tuning using the same validation set. Repeatedly selecting parameters according to one validation split could produce an overly optimistic estimate of performance.

Overall, **I treated AI recommendations as possible approaches to test, rather than as established conclusions**. I made the final model choice based on the experimental results obtained from my own Notebook.

I did not claim to have evaluated models or achieved competition scores that I had not actually observed.

**`From here on starts the code section for the competition.`**

---

# **Competition Code**

## 1. Preprocessing Steps

In [2]:

from pathlib import Path

# Check whether the four Kaggle files are available.
required_files = [
    "final_posts.json",
    "data_identification.csv",
    "emotion.csv",
    "sampleSubmission.csv",
]

search_dirs = [
    Path("."),
    Path("data"),
    Path("data/kaggle"),
]

data_paths = {}

for filename in required_files:
    matches = [
        folder / filename
        for folder in search_dirs
        if (folder / filename).is_file()
    ]

    if matches:
        data_paths[filename] = matches[0]
        print(f"FOUND: {filename} -> {matches[0]}")
    else:
        print(f"NOT FOUND: {filename}")

print(f"\nFiles found: {len(data_paths)}/{len(required_files)}")


FOUND: final_posts.json -> data/final_posts.json
FOUND: data_identification.csv -> data/data_identification.csv
FOUND: emotion.csv -> data/emotion.csv
FOUND: sampleSubmission.csv -> data/sampleSubmission.csv

Files found: 4/4


In [3]:

import pandas as pd

# Load the four Kaggle datasets.
posts = pd.read_json(data_paths["final_posts.json"])
identification = pd.read_csv(
    data_paths["data_identification.csv"]
)
emotions = pd.read_csv(data_paths["emotion.csv"])
sample_submission = pd.read_csv(
    data_paths["sampleSubmission.csv"]
)

# Match tweet text, train/test split, and labels by post_id.
df = posts.merge(
    identification,
    on="post_id",
    how="inner",
    validate="one_to_one"
)

df = df.merge(
    emotions,
    on="post_id",
    how="left",
    validate="one_to_one"
)

# Separate labeled training data from unlabeled test data.
train_df = df.loc[
    df["set"] == "train",
    ["post_id", "text", "label"]
].copy()

test_df = df.loc[
    df["set"] == "test",
    ["post_id", "text"]
].copy()

# Validate the dataset structure.
assert len(df) == 4800
assert len(train_df) == 3360
assert len(test_df) == 1440
assert train_df["label"].notna().all()
assert df.loc[df["set"] == "test", "label"].isna().all()

assert set(test_df["post_id"]) == set(
    sample_submission["post_id"]
)

print("Total tweets:", len(df))
print("Training samples:", len(train_df))
print("Test samples:", len(test_df))

print("\nTraining label distribution:")
print(train_df["label"].value_counts())

print("\nTraining data preview:")
display(train_df.head())

print("\nTest data preview:")
display(test_df.head())


Total tweets: 4800
Training samples: 3360
Test samples: 1440

Training label distribution:
label
joy         875
anger       674
sadness     628
disgust     454
surprise    441
fear        288
Name: count, dtype: int64

Training data preview:


,post_id,text,label
0,f8bv5zh,Thought it looked weird as well.,disgust
1,q0rr1l9,Bear died in a pit,disgust
2,p94rwgk,I have endured worse.,sadness
3,9u6wtvm,The pottery around Brasov is in danger,fear
4,h8ea329,"Meanwhile, our soldiers were approaching the a...",fear



Test data preview:


,post_id,text
3360,xyzx9jp,Damn. That’s profound. Did you graduate from H...
3361,l3vzzxt,When I said Lagos people were broke and noisy ...
3362,g680iug,Drunk and crying in bed with my pets. :|
3363,m4t8bp6,An afternoon of Big Cat vs Vol Twitter will be...
3364,ai3al6e,I still can't believe how quickly a little wat...


In [4]:

from sklearn.model_selection import StratifiedGroupKFold

# Group identical texts to prevent duplicate leakage.
groups = train_df["text"].str.strip().str.lower()

# Create a reproducible, approximately 80/20 split.
splitter = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

train_idx, val_idx = next(
    splitter.split(
        X=train_df["text"],
        y=train_df["label"],
        groups=groups
    )
)

train_part = train_df.iloc[train_idx].reset_index(drop=True)
val_part = train_df.iloc[val_idx].reset_index(drop=True)

# Check that identical normalized texts do not cross splits.
train_texts = set(
    train_part["text"].str.strip().str.lower()
)
val_texts = set(
    val_part["text"].str.strip().str.lower()
)

assert train_texts.isdisjoint(val_texts)

print("Training samples:", len(train_part))
print("Validation samples:", len(val_part))
print("Duplicate text overlap:", len(train_texts & val_texts))

print("\nTraining label distribution:")
print(train_part["label"].value_counts())

print("\nValidation label distribution:")
print(val_part["label"].value_counts())


Training samples: 2687
Validation samples: 673
Duplicate text overlap: 0

Training label distribution:
label
joy         700
anger       539
sadness     502
disgust     363
surprise    353
fear        230
Name: count, dtype: int64

Validation label distribution:
label
joy         175
anger       135
sadness     126
disgust      91
surprise     88
fear         58
Name: count, dtype: int64


## 2. Feature Engineering Steps

In [5]:

from sklearn.feature_extraction.text import TfidfVectorizer

# Word-level TF-IDF with unigrams and bigrams.
tfidf_word = TfidfVectorizer(
    lowercase=True,
    strip_accents="unicode",
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    max_features=30000,
    sublinear_tf=True
)

# Fit only on training data to prevent data leakage.
X_train_word = tfidf_word.fit_transform(
    train_part["text"].fillna("").astype(str)
)

# Reuse the fitted vectorizer for validation data.
X_val_word = tfidf_word.transform(
    val_part["text"].fillna("").astype(str)
)

assert X_train_word.shape[1] == X_val_word.shape[1]

print("Training feature matrix:", X_train_word.shape)
print("Validation feature matrix:", X_val_word.shape)
print("Vocabulary size:", len(tfidf_word.vocabulary_))


Training feature matrix: (2687, 6012)
Validation feature matrix: (673, 6012)
Vocabulary size: 6012


In [7]:

# Improved feature engineering:
# Character-level TF-IDF using 3- to 5-character n-grams.

tfidf_char = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(3, 5),
    min_df=2,
    max_features=60000,
    sublinear_tf=True
)

# Fit the vocabulary only on the training split.
X_train_char = tfidf_char.fit_transform(
    train_part["text"].fillna("").astype(str)
)

# Transform validation data without refitting.
X_val_char = tfidf_char.transform(
    val_part["text"].fillna("").astype(str)
)

assert X_train_char.shape[1] == X_val_char.shape[1]

print("Character TF-IDF training shape:", X_train_char.shape)
print("Character TF-IDF validation shape:", X_val_char.shape)
print("Character vocabulary size:", len(tfidf_char.vocabulary_))


Character TF-IDF training shape: (2687, 23360)
Character TF-IDF validation shape: (673, 23360)
Character vocabulary size: 23360


## 3. Model Implementation Steps

In [6]:

from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# Baseline: TF-IDF + Linear SVM.
baseline_model = LinearSVC(
    C=1.0,
    random_state=42,
    max_iter=5000
)

# Train using training data only.
baseline_model.fit(
    X_train_word,
    train_part["label"]
)

# Predict the held-out validation set.
val_pred_baseline = baseline_model.predict(X_val_word)

# Evaluate the baseline.
baseline_accuracy = accuracy_score(
    val_part["label"],
    val_pred_baseline
)

baseline_macro_f1 = f1_score(
    val_part["label"],
    val_pred_baseline,
    average="macro"
)

print(f"Validation Accuracy: {baseline_accuracy:.4f}")
print(f"Validation Macro-F1: {baseline_macro_f1:.4f}")

print("\nClassification Report:")
print(
    classification_report(
        val_part["label"],
        val_pred_baseline,
        digits=4,
        zero_division=0
    )
)

# Show the confusion matrix.
labels = sorted(train_part["label"].unique())

cm = confusion_matrix(
    val_part["label"],
    val_pred_baseline,
    labels=labels
)

cm_df = pd.DataFrame(
    cm,
    index=[f"True {label}" for label in labels],
    columns=[f"Pred {label}" for label in labels]
)

print("\nConfusion Matrix:")
display(cm_df)


Validation Accuracy: 0.4666
Validation Macro-F1: 0.4422

Classification Report:
              precision    recall  f1-score   support

       anger     0.4129    0.4741    0.4414       135
     disgust     0.3649    0.2967    0.3273        91
        fear     0.5405    0.3448    0.4211        58
         joy     0.5347    0.6171    0.5729       175
     sadness     0.4880    0.4841    0.4861       126
    surprise     0.4250    0.3864    0.4048        88

    accuracy                         0.4666       673
   macro avg     0.4610    0.4339    0.4422       673
weighted avg     0.4647    0.4666    0.4620       673


Confusion Matrix:


,Pred anger,Pred disgust,Pred fear,Pred joy,Pred sadness,Pred surprise
True anger,64,15,3,22,18,13
True disgust,25,27,5,15,11,8
True fear,9,6,20,11,9,3
True joy,17,13,3,108,20,14
True sadness,22,6,3,26,61,8
True surprise,18,7,3,20,6,34


In [8]:

# Improved model: Character TF-IDF + Linear SVM

char_model = LinearSVC(
    C=0.25,
    random_state=42,
    max_iter=5000
)

# Train on character-level TF-IDF features.
char_model.fit(
    X_train_char,
    train_part["label"]
)

# Predict the validation set.
val_pred_char = char_model.predict(X_val_char)

# Evaluate the improved model.
char_accuracy = accuracy_score(
    val_part["label"],
    val_pred_char
)

char_macro_f1 = f1_score(
    val_part["label"],
    val_pred_char,
    average="macro"
)

print(f"Validation Accuracy: {char_accuracy:.4f}")
print(f"Validation Macro-F1: {char_macro_f1:.4f}")

print("\nClassification Report:")
print(
    classification_report(
        val_part["label"],
        val_pred_char,
        digits=4,
        zero_division=0
    )
)

# Compare with the original baseline.
print("\n===== Model Comparison =====")
comparison_df = pd.DataFrame({
    "Model": [
        "Word TF-IDF + Linear SVM",
        "Character TF-IDF + Linear SVM"
    ],
    "Accuracy": [
        baseline_accuracy,
        char_accuracy
    ],
    "Macro-F1": [
        baseline_macro_f1,
        char_macro_f1
    ]
})

display(comparison_df)

print(
    f"\nMacro-F1 improvement: "
    f"{char_macro_f1 - baseline_macro_f1:+.4f}"
)


Validation Accuracy: 0.5557
Validation Macro-F1: 0.5342

Classification Report:
              precision    recall  f1-score   support

       anger     0.4643    0.5778    0.5149       135
     disgust     0.5814    0.2747    0.3731        91
        fear     0.8182    0.4655    0.5934        58
         joy     0.5982    0.7657    0.6717       175
     sadness     0.5115    0.5317    0.5214       126
    surprise     0.5811    0.4886    0.5309        88

    accuracy                         0.5557       673
   macro avg     0.5924    0.5174    0.5342       673
weighted avg     0.5695    0.5557    0.5466       673


===== Model Comparison =====


,Model,Accuracy,Macro-F1
0,Word TF-IDF + Linear SVM,0.466568,0.442244
1,Character TF-IDF + Linear SVM,0.555721,0.534223



Macro-F1 improvement: +0.0920


In [9]:

# Final model: train on all labeled Kaggle data.
# Model settings were selected using the validation split.

final_tfidf = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(3, 5),
    min_df=2,
    max_features=60000,
    sublinear_tf=True
)

# Fit on all 3,360 labeled training samples.
X_full_train = final_tfidf.fit_transform(
    train_df["text"].fillna("").astype(str)
)

final_model = LinearSVC(
    C=0.25,
    random_state=42,
    max_iter=5000
)

final_model.fit(
    X_full_train,
    train_df["label"]
)

# Transform the 1,440 unlabeled Kaggle test samples.
X_kaggle_test = final_tfidf.transform(
    test_df["text"].fillna("").astype(str)
)

test_predictions = final_model.predict(X_kaggle_test)

# Match predictions to the sample submission by post_id.
prediction_df = test_df[["post_id"]].copy()
prediction_df["label"] = test_predictions

submission = sample_submission[["post_id"]].merge(
    prediction_df,
    on="post_id",
    how="left",
    validate="one_to_one",
    sort=False
)

# Verify submission integrity.
assert len(submission) == 1440
assert submission["label"].notna().all()
assert submission["post_id"].tolist() == (
    sample_submission["post_id"].tolist()
)
assert set(submission["label"]).issubset(
    set(train_df["label"])
)

# Export predictions in Kaggle CSV format.
output_path = "kaggle_submission_char_svm.csv"
submission.to_csv(output_path, index=False)

print("Final training samples:", len(train_df))
print("Kaggle test predictions:", len(submission))
print("Final vocabulary size:", len(final_tfidf.vocabulary_))
print("Submission saved to:", output_path)

print("\nPredicted label distribution:")
print(submission["label"].value_counts())

print("\nSubmission preview:")
display(submission.head())


Final training samples: 3360
Kaggle test predictions: 1440
Final vocabulary size: 26521
Submission saved to: kaggle_submission_char_svm.csv

Predicted label distribution:
label
joy         527
anger       345
sadness     247
surprise    150
disgust      99
fear         72
Name: count, dtype: int64

Submission preview:


,post_id,label
0,xyzx9jp,anger
1,l3vzzxt,fear
2,g680iug,sadness
3,m4t8bp6,joy
4,ai3al6e,surprise
